# PyTorch

## torch模块
torch模块 torch.nn.module 是PyTorch的核心模块，提供了张量（Tensor）操作、自动求导（Autograd）功能以及神经网络（nn）模块等。它是PyTorch中最基础的模块，几乎所有的PyTorch代码都会使用到torch模块。

### PyTorch模型核心术语
包括 `nn.Parameter`, `register_buffer`, `model.parameters`, `model.state_dicr()`, `model.to(device)`等

`nn.Parameter` （可学习参数）
> 模型中需要通过反向传播更新的张量。例如线性层的权重 $W$ 和偏置 $b$。
```python
self.weight = nn.Parameter(torch.randn(3, 3))  # 会被优化器更新
```
创建为 `nn.Parameter` 后，PyTorch 自动：

标记 requires_grad=True（开启梯度追踪），纳入 model.parameters() 列表

而在梯度计算时，如
```python
loss = criterion(model(x), y)   # 前向传播，计算损失
loss.backward()                  # 反向传播，计算每个 Parameter 的梯度 ∂L/∂W
optimizer.step()                 # 用梯度更新参数
```
nn.Parameter → 参与这个过程，会被 backward() 计算梯度，被 optimizer.step() 更新，register_buffer → 不参与，梯度不会流过它，优化器也不碰它。

`model.parameters`（参数迭代器）
> 返回模型中所有 nn.Parameter 的迭代器。最主要的用途是传给优化器：
```python
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
#                            ^^^^^^^^^^^^^^^^
#                            告诉优化器：这些张量需要更新
```
register_buffer 注册的张量不在其中，所以优化器不会去动它们

`model.state_dict()`（模型状态字典）
> 模型的完整快照，是一个 {名称: 张量} 的有序字典。用于保存和加载模型：
```python
# 保存
torch.save(model.state_dict(), "checkpoint.pth")
# 加载
model.load_state_dict(torch.load("checkpoint.pth"))
```
它包含所有 nn.Parameter（权重、偏置等），所有 register_buffer（如 $\beta_t$ 调度序列）

`model.to(device)` (设备迁移)
> 把模型所有张量从 CPU 移到 GPU（或反过来）：
```python
model = model.to("cuda")  # 或 model.cuda()
```
这一步会移动：所有 nn.Parameter 和所有 register_buffer
不会移动普通属性 self.xxx = some_tensor。如果忘了用 register_buffer，训练时就会遇到经典报错：

> RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

**小结：**
```bash
model
├── nn.Parameter (权重W, 偏置b, ...)
│   ├── 有梯度 ✓
│   ├── optimizer 更新 ✓
│   ├── state_dict 包含 ✓
│   └── .to(device) 跟随 ✓
│
├── register_buffer (betas, alpha_bars, ...)
│   ├── 无梯度 ✗
│   ├── optimizer 不碰 ✗
│   ├── state_dict 包含 ✓    ← 会被保存/加载
│   └── .to(device) 跟随 ✓    ← 自动移设备
│
└── 普通属性 (hidden_dim=256, mode="train", ...)
    ├── 无梯度 ✗
    ├── state_dict 不包含 ✗
    └── .to(device) 不跟随 ✗
```